# BrainTumorAI Google Colab Backend
This notebook runs the BrainTumorAI FastAPI server directly on Google Colab's GPU.
**Instructions:**
1. Ensure you are using a **T4 GPU** runtime (Runtime -> Change runtime type -> Hardware accelerator: T4 GPU).
2. Press **Run all** (Cmd/Ctrl + F9). The notebook will automatically clone the codebase, setup models, and start the API.

In [ ]:
import os

# Clone the GitHub repository automatically if it does not exist
if not os.path.exists('/content/BrainTumorAI'):
    !git clone https://github.com/BiswasApurbo/BrainTumorAI.git /content/BrainTumorAI
else:
    print("Repository already exists. Pulling latest changes...")
    !cd /content/BrainTumorAI && git pull

%cd /content/BrainTumorAI


In [ ]:
# Install uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] += ":/root/.cargo/bin"

# Sync dependencies
!uv sync


In [ ]:
import os
import sys
import torch

local_models_path = '/content/BrainTumorAI/models'
nnunet_model_path = os.path.join(local_models_path, 'nnUNet')
synthseg_model_path = os.path.join(local_models_path, 'SynthSeg')

# 1. Environment Variables
os.environ['RESULTS_FOLDER'] = local_models_path
os.environ['nnUNet_raw_data_base'] = os.path.join(local_models_path, 'nnUNet_raw_data_base')
os.environ['nnUNet_preprocessed'] = os.path.join(local_models_path, 'nnUNet_preprocessed')

# 2. Verification & Startup Summary
print("\n====================================")
print("         STARTUP SUMMARY            ")
print("====================================")
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")
if cuda_available:
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    
nnunet_found = os.path.exists(nnunet_model_path)
synthseg_found = os.path.exists(synthseg_model_path)
print(f"nnUNet Models Found: {nnunet_found}")
print(f"SynthSeg Models Found: {synthseg_found}")
print("====================================\n")

if not cuda_available:
    raise RuntimeError("CUDA is not available. Please ensure you are using a GPU runtime (T4).")
if not nnunet_found:
    raise FileNotFoundError("nnUNet models are missing from the cloned repository.")
if not synthseg_found:
    raise FileNotFoundError("SynthSeg models are missing from the cloned repository.")


In [ ]:
!pip install pyngrok -q
from pyngrok import ngrok

# Replace with your ngrok authtoken if you want to use a stable account
# ngrok.set_auth_token("YOUR_AUTHTOKEN")

# Open a HTTP tunnel on the default port 8000
public_url = ngrok.connect(8000)
print(f"\n\n>>> OPEN THIS URL IN YOUR LOCAL BROWSER: {public_url} <<<\n\n")

In [ ]:
# Start the FastAPI server
!PYTHONPATH=src uv run uvicorn backend.main:app --host 0.0.0.0 --port 8000
